# 03 · Score everything, build the tables and figures

**CPU runtime** — no GPU. The geometry kernel is the bottleneck, not the GPU.

This notebook turns raw model outputs into meshes, scores every mesh under the single
shared protocol, and emits the six tables and three figures.

In [ ]:
# Mount Drive so predictions survive a Colab timeout, then get the harness.
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/t2c_bench'
os.makedirs(WORK, exist_ok=True)
os.environ['T2C_WORK'] = WORK

!git clone -q https://github.com/prashantkul366/T2C_Benchamrk /content/t2cbench_repo || (cd /content/t2cbench_repo && git pull -q)
%cd /content/t2cbench_repo
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
print('work dir:', WORK)

In [ ]:
# CPU-side evaluation dependencies. embreex matters: without it the exact
# point-in-solid test falls back to a pure-Python ray engine and voxel IoU goes
# from ~0.5s to ~90s per sample.
!pip install -q trimesh rtree embreex manifold3d scipy pandas matplotlib tabulate pyyaml tqdm
!pip install -q cadquery
import trimesh, embreex
print('trimesh', trimesh.__version__, '| embreex present')

### pythonocc-core

Needed only by the sequence adapters (Text2CAD, CADmium, CADFusion) — CadQuery uses OCP
instead. On Colab, conda is the reliable route.

In [ ]:
!pip install -q condacolab
import condacolab; condacolab.install()   # this restarts the runtime; re-run the cells above after it

In [ ]:
!mamba install -q -y -c conda-forge pythonocc-core=7.7.0
!git clone -q --depth 1 https://github.com/SadilKhan/Text2CAD /content/Text2CAD || true
!git clone -q --depth 1 https://github.com/microsoft/CADFusion /content/CADFusion || true
import os
os.environ['T2CBENCH_CADSEQ_PATH'] = '/content/Text2CAD'
os.environ['T2CBENCH_CADFUSION_PATH'] = '/content/CADFusion'

### Score every raw prediction file

The adapter is chosen from `configs/models.yaml`, so each system is scored through its own
output format but against the *same* metrics. Invalid outputs are scored as failures, never
dropped — that is the whole point.

In [ ]:
import glob, os, yaml, re
WORK = os.environ['T2C_WORK']
models = yaml.safe_load(open('configs/models.yaml'))

for raw in sorted(glob.glob(f'{WORK}/results/raw/*.jsonl')):
    stem = os.path.basename(raw)[:-6]
    name = re.split(r'_split[AB]', stem)[0]
    key  = name.replace('_0shot', '')
    adapter = models.get(key, {}).get('adapter', 'cadquery')
    split_letter = 'a' if '_splitA' in stem else 'b'
    scored = f'{WORK}/results/scored/{stem}.jsonl'
    if os.path.exists(scored):
        print('skip (done):', stem); continue
    print(f'>>> {stem}  adapter={adapter}')
    !python -m t2cbench.evaluate \
        --predictions {raw} \
        --split {WORK}/data/split_{split_letter}.jsonl \
        --adapter {adapter} --model-name {name} \
        --keep-meshes {WORK}/results/meshes/{name} \
        --out {scored} --workers 8 --timeout 20

### Tables

In [ ]:
!python -m t2cbench.report.tables \
    --scored '{WORK}/results/scored/*.jsonl' \
    --out {WORK}/results/tables \
    --ablation-pairs configs/ablation_pairs.json

from IPython.display import Markdown, display
display(Markdown(open(f'{WORK}/results/tables/RESULTS.md').read()))

### Figures

In [ ]:
!python -m t2cbench.report.figures \
    --scored '{WORK}/results/scored/*.jsonl' \
    --out {WORK}/results/figures \
    --split {WORK}/data/split_a.jsonl \
    --mesh-root {WORK}/results/meshes \
    --families configs/model_families.json

from IPython.display import Image, display
for f in ['fig1_failure_modes','fig2_level_sensitivity','fig3_qualitative']:
    display(Image(f'{WORK}/results/figures/{f}.png'))

### Reading the results

* **Score = P(valid) × F1@0.02** is the ranking column. It cannot be gamed by answering few
  prompts well, which median-CD-over-valid-only can.
* **Compare published numbers against the best-of-5 table, not pass@1.** Text2CAD, CADFusion
  and cadrille all report oracle best-of-N.
* **The contamination table is a memorisation probe**, not a leaderboard — n=13 clean
  CADPrompt objects is far too small to rank on. The clean↔contaminated *gap* per model is
  the number to read.
* **CADFusion carries a "split unverified" caveat**: it trains on SkexGen, whose train/test
  boundary is not DeepCAD's, so some Split A shapes may be in its training data.